# Demonstration 3: a reversible bimolecular reaction in a one-body ramp potential

**What this validates:** the transport layer and the Fröhner–Noé acceptance rule,
with volume exclusion switched off.

**Analytic answer:** for a reversible reaction $A + B \rightleftharpoons C$ in a
static potential, the *local* equilibrium quotient follows the reaction's field work:

$$Q(x) \;=\; \frac{\langle n_C(x)\rangle}{\langle n_A n_B\rangle(x)}
  \;=\; \frac{k_F}{k_R}\, e^{-\Delta\phi(x)},
  \qquad \Delta\phi(x) = \sum_s \Delta\nu_s\,\phi_s(x)$$

Only $C$ is coupled to the field here ($\gamma_C = g$, $\gamma_A = \gamma_B = 0$), so
$\Delta\phi(x) = g\,\psi(x)$.

Note the denominator is $\langle n_A n_B\rangle$, the mean of the *product*, not the
product of the means. $A$ and $B$ are correlated through the conservation law, and only
the mean of the product satisfies the detailed-balance relation.

## Symbols, and where they live in the code

Every symbol used below, with the variable that carries it. Energies are in units of
$k_BT$ throughout, so $\beta = 1/k_BT$ never appears explicitly: what the code calls a
free energy is already $\beta F$.

### Lattice and transport

| symbol | code | meaning |
|---|---|---|
| $d$ | `lattice.dim` | dimension, 2 or 3 |
| $h$ | `voxel_nm` | voxel edge length, nm |
| $V$ | `lattice.voxel_volume_nm3` | voxel volume $h^3$, nm$^3$ (cubic even in 2D) |
| $\tau$ | `tau_s` | timestep, s |
| $D$ | `D_um2_s` | diffusion coefficient, $\mu$m$^2$/s |
| $q$ | `sim.hop.q` | baseline per-direction hop probability, $q = D\tau/h^2$ |
| $n_s(v)$ | `state.counts[s, v]` | integer count of species $s$ in voxel $v$ |
| $\rho$ | | number density, here the mean occupancy per voxel |

### Fields and coupling

| symbol | code | meaning |
|---|---|---|
| $\psi_k(v)$ | `psi[k]` | static basis field $k$, supplied by you and never modified |
| $\gamma_{s,k}$ | `Species.gamma` | coupling of species $s$ to field $k$ |
| $\phi_s(v)$ | | the potential species $s$ feels, $\phi_s(v) = \sum_k \gamma_{s,k}\,\psi_k(v)$ |

### Hard spheres

| symbol | code | meaning |
|---|---|---|
| $\sigma_s$ | `sigma_nm` | hard-sphere diameter of species $s$, nm |
| $d\xi^{(k)}_s$ | `exclusion.dxi[s, k]` | per-particle increment, $d\xi^{(k)}_s = \frac{\pi}{6}\sigma_s^k / V$ |
| $\xi_k(v)$ | `exclusion.xi(counts)[k]` | weighted density $k$, for $k = 0,1,2,3$ |
| $\eta$ | | packing fraction, the same thing as $\xi_3$ |
| $\beta F_{\rm ex}$ | `vex.bfex(xi, V)` | White-Bear/BMCSL excess free energy of a voxel |
| $\mu_{\rm ex}$ | `mu_ex_carnahan_starling` | excess chemical potential, $k_BT$ |

### The formulas the code implements

**Weighted densities.** Fundamental-measure theory reduces a hard-sphere mixture to
four scalar fields. On a lattice each is linear in the integer counts:

$$\xi_k(v) = \sum_s n_s(v)\, d\xi^{(k)}_s,
\qquad d\xi^{(k)}_s = \frac{\pi}{6}\frac{\sigma_s^{\,k}}{V},
\qquad k = 0,1,2,3.$$

$\xi_3$ is the packing fraction. $\xi_0$, $\xi_1$ and $\xi_2$ carry number, radius and
surface, which is why a reaction that merges two spheres into one of equal *volume*
still changes the free energy.

**White-Bear / BMCSL excess free energy**, per voxel, in $k_BT$:

$$\beta F_{\rm ex} = V\left[
  -\frac{6}{\pi}\,\xi_0 \ln(1-\xi_3)
  + \frac{18}{\pi}\,\frac{\xi_1 \xi_2}{1-\xi_3}
  + \frac{6}{\pi}\,\frac{\xi_2^3}{\xi_3^2}
    \left(\frac{\xi_3}{(1-\xi_3)^2} + \ln(1-\xi_3)\right)\right].$$

The $\ln(1-\xi_3)$ in the third term is what makes this BMCSL rather than the
Rosenfeld-1989 form. The factor $V$ converts a free-energy density into a per-voxel
energy.

**The hop.** A particle of species $s$ moves from voxel $v$ to a neighbour
$v' = v + e_\delta$ with probability

$$p_\delta(v) = q\; B\!\left(u_\delta(v)\right),
\qquad B(u) = \frac{u}{e^u - 1},$$

where $B$ is the Scharfetter-Gummel (Wang-Peskin-Elston) factor and $u$ is the total
work of the move in $k_BT$:

$$u_\delta(v) = \underbrace{\phi_s(v') - \phi_s(v)}_{\text{field}}
  + \underbrace{\Big[F(\xi_{v'} + d\xi_s) - F(\xi_{v'})\Big]}_{\text{insertion at } v'}
  - \underbrace{\Big[F(\xi_{v}) - F(\xi_{v} - d\xi_s)\Big]}_{\text{removal at } v}.$$

The $-d\xi_s$ in the last bracket is the **self-exclusion**: the hopping particle must
not feel its own volume at the voxel it is leaving. $B(0) = 1$ exactly, so with no work
the hop probability is just $q$.

**Why $\tau$ is small.** $B(u) \to |u|$ as $u \to -\infty$, so a strongly downhill move
can carry a probability far above $q$. The per-direction probabilities plus the stay
probability must sum to at most 1, which gives

$$2\,d\,q\,\mu_{\rm ex}(\eta_{\max}) \le 1$$

rather than the bare $q \le 1/(2d)$. At $\eta = 0.5$, $\mu_{\rm ex} \approx 17\,k_BT$,
so the admissible $\tau$ is an order of magnitude below the CFL bound.
`suggest_tau` computes it.

### Reactions

| symbol | code | meaning |
|---|---|---|
| $k_F$, $k_R$ | `k_forward`, `k_reverse` | forward and reverse rate constants |
| $\Delta\nu_s$ | `Reaction.dnu` | change in the count of species $s$ when the reaction fires once |
| $\Delta\Phi$ | | total work of the reaction, field plus exclusion, in $k_BT$ |
| $\pi_F$, $\pi_R$ | `ReactionSet.acceptance` | Fröhner-Noé acceptance factors |
| $Q$ | | reaction quotient, $\langle n_C\rangle / \langle n_A n_B\rangle$ |
| $K_{\rm eq}$ | | equilibrium constant, the value $Q$ takes at equilibrium |

**Propensity and acceptance.** A reaction of order 2 has per-voxel propensity
$k_F\, n_A(v)\, n_B(v)$, multiplied by an acceptance factor

$$\pi = \min\!\left(1,\; e^{-\Delta\Phi}\right),
\qquad \Delta\Phi = \underbrace{\sum_s \Delta\nu_s\, \phi_s(v)}_{\text{field}}
  + \underbrace{\Big[F(n + \Delta\nu) - F(n)\Big]}_{\text{exclusion}}.$$

**Reversibility.** The reverse work is evaluated from the **post-reaction** count
vector, not the pre-reaction one:

$$\Delta\Phi_F \text{ from } n, \qquad \Delta\Phi_R \text{ from } n + \Delta\nu
\quad\Longrightarrow\quad \Delta\Phi_R = -\Delta\Phi_F .$$

That makes $\pi_F/\pi_R = e^{-\Delta\Phi_F}$ identically, so detailed balance holds by
construction rather than approximately. Evaluating the reverse from $n$ instead gives
$F(n - \Delta\nu) - F(n)$, which is not the negative of the forward work, and detailed
balance then breaks silently.

**Note on the quotient.** The denominator is $\langle n_A n_B\rangle$, the mean of the
product, not $\langle n_A\rangle\langle n_B\rangle$. The reactants are correlated
through the conservation law, and only the mean of the product satisfies the
detailed-balance relation.

### Extra symbols for this demonstration

| symbol | code | meaning |
|---|---|---|
| $x$ | | position along the field axis, in voxels |
| $Q(x)$ | `QuotientAccumulator` | reaction quotient resolved along that axis |
| $g$ | `GAMMA_C` | the coupling $\gamma_C$ of the product to the ramp |

The potential is a **one-body ramp**: a single basis field rising linearly along one
axis, $\psi(x) = x / L$, with only the product coupled to it. So
$\phi_A = \phi_B = 0$ and $\phi_C(x) = g\,\psi(x)$, which makes the reaction work
purely a field term:

$$\Delta\Phi(x) = \sum_s \Delta\nu_s\,\phi_s(x) = g\,\psi(x).$$

Volume exclusion is switched off here, so $F_{\rm ex}$ contributes nothing and this
demonstration isolates transport and acceptance. The prediction is then

$$Q(x) = \frac{\langle n_C(x)\rangle}{\langle n_A n_B\rangle(x)}
       = \frac{k_F}{k_R}\, e^{-g\,\psi(x)}.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vex_rddme import Simulation, Species, mu_ex_carnahan_starling
from vex_rddme.guards import suggest_tau
from vex_rddme import viz
from vex_rddme.observe import (
    Series, project, mu_ex_from_profile, align_additive_constant,
    relative_discrepancy, report_comparison, QuotientAccumulator,
)

## Setup

A 2D lattice with a linear ramp in $\psi$ along the short axis. Exclusion is off, so
this demonstration isolates transport and acceptance. If it fails, the fault is in one of those
two and not in the free-energy machinery.

In [ ]:
SHAPE      = (32, 24)     # (rows, field axis)
VOXEL_NM   = 20.0
GAMMA_C    = 1.5         # only C feels the field
K_F, K_R   = 60.0, 60.0
N_A = N_B  = 3000
TAU_S      = 2.0e-5      # exclusion off, so the bare CFL bound applies
N_STEPS    = 30_000
BURN_IN    = N_STEPS // 3
SAMPLE_EVERY = 25        # samples are correlated; space them out

ramp = np.arange(SHAPE[-1], dtype=float) / SHAPE[-1]
psi  = np.broadcast_to(ramp, SHAPE).copy()[None, ...]

sim = Simulation(
    shape=SHAPE, voxel_nm=VOXEL_NM,
    species=[
        Species("A", sigma_nm=0.0, gamma=np.array([0.0])),
        Species("B", sigma_nm=0.0, gamma=np.array([0.0])),
        Species("C", sigma_nm=0.0, gamma=np.array([GAMMA_C])),
    ],
    occupancy_cap=400, psi=psi, D_um2_s=1.0, tau_s=TAU_S,
    exclusion=False, seed=0,
)
sim.add_reaction("assoc", ["A", "B"], ["C"], K_F, K_R, typical_reactant_product=9.0)
sim.seed_uniform("A", N_A)
sim.seed_uniform("B", N_B)
sim.record_initial()
sim

## Run

Every guard is live during the run. If the timestep or the rate constants were wrong
for this configuration, the run would stop and say so rather than quietly producing a
plausible-looking wrong answer.

In [ ]:
acc = QuotientAccumulator(
    reactants=(0, 1), products=(2,), lattice=sim.lattice, axis=-1
)

for i in range(N_STEPS):
    sim.step()
    if i >= BURN_IN and (i - BURN_IN) % SAMPLE_EVERY == 0:
        acc.add(sim.state.counts)

sim.state.check_mass()          # exact: integers in, integers out
print(f"{acc.n} samples over {N_STEPS - BURN_IN} steps")
print("totals A, B, C:", sim.state.totals().tolist())
print("mean acceptance  forward %.4f   reverse %.4f"
      % (sim.reactions.mean_acceptance(0, 0), sim.reactions.mean_acceptance(0, 1)))

## Compare against the analytic prediction

Both curves are normalised by their means, so the comparison is of *shape*: the
absolute level is set by $k_F/k_R$, which this demonstration is not testing (that is the
well-mixed check in the test suite).

In [ ]:
Q_measured  = acc.quotient
Q_sem       = acc.sem
Q_predicted = (K_F / K_R) * np.exp(-GAMMA_C * ramp)

m = Q_measured  / Q_measured.mean()
p = Q_predicted / Q_predicted.mean()
s = Q_sem       / Q_measured.mean()

print(report_comparison("Q(x) shape vs (k_F/k_R) exp(-g psi(x))", m, p, sem=s))

# Exposed so the test suite can check this notebook's claim without a human reading it.
VERDICTS = {"Q(x) shape vs exp(-dPhi)": relative_discrepancy(m, p)["max"]}

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
viz.plot_profile(m, predicted=p, sem=s, ax=ax[0],
                 xlabel="voxel along the field axis", ylabel="Q(x) / mean",
                 label_measured="measured", label_predicted=r"$e^{-g\psi}$ (normalised)",
                 title="Demonstration 3: local equilibrium follows the field")
viz.show_lattice(sim.state.counts[2], sim.lattice, ax=ax[1], title="C occupancy")
plt.tight_layout(); plt.show()

## What to take from this

The measured quotient tracks $e^{-\Delta\phi(x)}$ across the box. Because $A$ and $B$
are uncoupled from the field, their densities stay flat. So the entire spatial
structure in $Q(x)$ comes from the acceptance rule, not from reactant transport.

**Try changing:**

- `GAMMA_C = 0`: the prediction becomes flat, and so should the measurement.
- `GAMMA_C = 4`: a steeper field. Watch for the `hop-probability-sum` guard: the
  Bernoulli factor grows for downhill moves and the timestep may no longer be small
  enough. That is the guard doing its job, not a bug.
- `K_F = 6.0`: a hundredfold slower reaction. The measurement will not have
  equilibrated in `N_STEPS`, and the discrepancy will grow. Reaction equilibration time
  scales as $1/k$.